**Caso de estudio-integracion vertical eps**
Una eps quiere hacer una optimizacion de sus operaciones por lo que quiere hacer una integracion de sus diferentes sucursales ubicadas en el valle de aburra. en este senitdo, la eps quiere centrar sus operaciones en los secrtores mas cercanos a la cuidad de medellin por lo que quiere hacer la integracion de los pacientes de copacaba y caldas en sus otras sedes. Dentro de  los obejticos, quiere evaluar como era la configuracion de sus variables para antes y despus de la itnegracion de lso paciente nuevos.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import MinMaxScaler
from google.colab import drive
drive.mount('/content/drive')

1. cargamos la base de datos de referencia y creamos las bases de datos de integracion

In [ ]:
nxl="/content/drive/MyDrive/Integracion de datos/5. Diabetes Árbol_Int_Mult.xlsx"
XDB=pd.read_excel(nxl,sheet_name=0)
XDB.dropna()
XDB.head()

# Se construye la primera base de datos - Solo con variables de entrada
XD=XDB.iloc[:,[0,1,2,3,4,5,6,7,9,10]]
XD.head()

# Se construye la segunda base de datos
XDB2=XDB.iloc[:,[0,1,2,3,4,5,6,7,9,10,11]]
XDB2.head()

2. Se pocede con la cercion de los clusters

In [ ]:
import numpy as np
import pandas

# Selecciono las variables que se usarán para crear los clusters
XD = XDB.iloc[:, [0,1,2,3,4,5,6,7,9,10]]

# Inicializo el vector donde se guardará el cluster de cada dato
nc = np.zeros((len(XD),1))

# Convierto los datos a NumPy
XD_array = np.array(XD)

# Semillas iniciales
XC = XD_array[0:5,:].copy()

# -----------------------------
# CREACIÓN DE LOS CLUSTERS
# -----------------------------

for k in range(len(XD_array)):

    # Distancia euclidiana
    d = np.sqrt(
        np.sum((XC - XD_array[k,:])**2, axis=1)
    )

    # Cluster más cercano
    cl = int(np.argmin(d))

    # Guardo el cluster
    nc[k,0] = cl

    # Actualizo el centro
    XC[cl,:] = (XC[cl,:] + XD_array[k,:]) / 2


# -----------------------------
# CENTROS DE LOS CLUSTERS
# -----------------------------

centros = pandas.DataFrame(
    XC,
    columns=XD.columns
)

print("Centros de los clusters:")
display(centros)


# -----------------------------
# AGREGO CLUSTER A LA BASE
# -----------------------------

XDB2 = XDB.copy()

XDB2["Cluster"] = nc.astype(int).ravel()

print("Base con cluster:")
display(XDB2.head())


# -----------------------------
# PERSONAS CON DIABETES
# -----------------------------

NPD = XDB2.groupby("Cluster").agg(
    {"Outcome": lambda x: (x == 1).sum()}
)

print("Personas con diabetes:")
display(NPD)


# -----------------------------
# PERSONAS SIN DIABETES
# -----------------------------

NPND = XDB2.groupby("Cluster").agg(
    {"Outcome": lambda x: (x == 0).sum()}
)

print("Personas sin diabetes:")
display(NPND)


# -----------------------------
# PROBABILIDADES
# -----------------------------

P_Diabetes = NPD / (NPD + NPND)

P_No_Diabetes = NPND / (NPD + NPND)

print("Probabilidad de diabetes:")
display(P_Diabetes)

print("Probabilidad de no diabetes:")
display(P_No_Diabetes)


# -----------------------------
# RESULTADO FINAL
# -----------------------------

resultado = pandas.DataFrame(
    np.column_stack(
        (
            XC,
            P_Diabetes.values,
            P_No_Diabetes.values
        )
    ),
    columns=list(XD.columns) +
            ["P_Diabetes", "P_No_Diabetes"]
)

print("Resultado final:")
display(resultado)

**Analisis de resultados**
En el segundo clúster se puede observar el mayor porcentaje de personas con diabetes, esto promovido principalmente por un nivel de insulina particularmente bajo (50.89). Se destaca en este mismo un número elevado de atenciones por año para cada paciente (6.21), lo que refuerza la presencia de diabetes en este grupo de pacientes. Se destaca el clúster 5, el cual posee el porcentaje más alto de pacientes con diabetes, esto promovido por un valor de insulina más bajo para todos los grupos y con edades mucho más altas.